In [ ]:
[
  {
    "prompt": "Explain what machine learning is in simple words.",
    "chosen": "Machine learning is a way to teach computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every case.",
    "rejected": "Machine learning is a machine that automatically learns everything without data."
  },
  {
    "prompt": "What is overfitting in machine learning?",
    "chosen": "Overfitting happens when a model memorizes the training data too closely and performs poorly on new unseen data.",
    "rejected": "Overfitting means the model becomes perfect for all types of data."
  },
  {
    "prompt": "Difference between classification and regression?",
    "chosen": "Classification predicts categories or labels, while regression predicts continuous numerical values.",
    "rejected": "Classification and regression are exactly the same thing."
  },
  {
    "prompt": "What is the purpose of a loss function?",
    "chosen": "A loss function measures how far a model’s predictions are from the true values, guiding the model during training.",
    "rejected": "A loss function increases model size and does not affect training."
  }
]

In [ ]:
!pip install torch transformers datasets trl peft accelerate bitsandbytes sentencepiece

In [ ]:
import os
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# DATA_PATH = "data/preference_data.json"
OUTPUT_DIR = "outputs/dpo_model"

In [ ]:
# with open(DATA_PATH, "r", encoding="utf-8") as f:
#     data = json.load(f)

# dataset = Dataset.from_list(data)

# print("Sample training row:")
# print(dataset[0])

In [ ]:
from datasets import load_dataset
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

In [ ]:
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=20,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
    max_length=512,
    beta=0.1,
    loss_type="sigmoid"
)

In [ ]:
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Training completed. Model saved to: {OUTPUT_DIR}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "outputs/dpo_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

prompt = "Explain overfitting in machine learning."

messages = [
    {"role": "user", "content": prompt}
]

try:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
except Exception:
    text = prompt

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)